# Capítulo 9 · Del modelo base al asistente

Ajuste supervisado del GPT del capítulo 7 con conversaciones, máscara de la pérdida, mezcla de texto para no olvidar la novela, evaluación antes y después y rondas de recompensas. PyTorch en CPU. El cuaderno completo tarda unos nueve minutos.

La explicación completa está en el capítulo en PDF. Este cuaderno es el laboratorio.

## 1. Constantes, tareas y modelo

En Colab, la celda descarga el modelo del capítulo 7 y la novela si no están.

In [1]:
"""Capítulo 9: del modelo base al asistente. PyTorch en CPU."""
import math
import random
import re
import time
from pathlib import Path

import torch
from torch import nn
from torch.nn import functional as F

RAIZ = Path(".")
MODELO = RAIZ / "material" / "gpt_quijote.pt"
TEXTO = RAIZ / "material" / "quijote.txt"
for archivo in (MODELO, TEXTO):  # en Colab, los descargamos del repositorio
    if not archivo.exists():
        import urllib.request
        archivo.parent.mkdir(exist_ok=True)
        urllib.request.urlretrieve("https://raw.githubusercontent.com/HernanDiaz/language-models/main/material/" + archivo.name, archivo)
SEMILLA = 1337
DEMOS_REVES = 300  # respuestas escritas para «escribe al revés»
MEZCLAR = True     # añadir texto de la novela a cada lote del ajuste
TAREAS = {
    "repite": lambda palabra: palabra,
    "pon en mayúsculas": lambda palabra: palabra.upper(),
    "escribe al revés": lambda palabra: palabra[::-1],
}


# --- El modelo de los capítulos 6 a 8 ---------------------------------------

class Cabeza(nn.Module):
    def __init__(self, config):
        super().__init__()
        C, T = config["dimension"], config["contexto"]
        tamano = C // config["cabezas"]
        self.consulta = nn.Linear(C, tamano, bias=False)
        self.clave = nn.Linear(C, tamano, bias=False)
        self.valor = nn.Linear(C, tamano, bias=False)
        self.register_buffer("mascara", torch.tril(torch.ones(T, T)))

    def forward(self, x):
        T = x.shape[1]
        q, k, v = self.consulta(x), self.clave(x), self.valor(x)
        puntuaciones = q @ k.transpose(1, 2) / math.sqrt(k.shape[-1])
        futuro = self.mascara[:T, :T] == 0
        puntuaciones = puntuaciones.masked_fill(futuro, float("-inf"))
        return F.softmax(puntuaciones, dim=-1) @ v


class Bloque(nn.Module):
    def __init__(self, config):
        super().__init__()
        C = config["dimension"]
        self.cabezas = nn.ModuleList(
            [Cabeza(config) for _ in range(config["cabezas"])])
        self.proyeccion = nn.Linear(C, C)
        self.mlp = nn.Sequential(nn.Linear(C, 4 * C), nn.ReLU(),
                                 nn.Linear(4 * C, C))
        self.norma1, self.norma2 = nn.LayerNorm(C), nn.LayerNorm(C)

    def forward(self, x):
        h = self.norma1(x)
        atencion = torch.cat([cabeza(h) for cabeza in self.cabezas], dim=-1)
        x = x + self.proyeccion(atencion)
        return x + self.mlp(self.norma2(x))


class GPT(nn.Module):
    def __init__(self, vocabulario, config):
        super().__init__()
        C = config["dimension"]
        self.config = config
        self.embedding_simbolos = nn.Embedding(vocabulario, C)
        self.embedding_posiciones = nn.Embedding(config["contexto"], C)
        self.bloques = nn.Sequential(
            *[Bloque(config) for _ in range(config["capas"])])
        self.norma = nn.LayerNorm(C)
        self.salida = nn.Linear(C, vocabulario)

    def forward(self, ids):
        posiciones = torch.arange(ids.shape[1])
        x = self.embedding_simbolos(ids) + self.embedding_posiciones(posiciones)
        return self.salida(self.norma(self.bloques(x)))


def cargar_modelo(ruta=MODELO):
    guardado = torch.load(ruta)
    simbolos = guardado["simbolos"]
    modelo = GPT(len(simbolos), guardado["config"])
    modelo.load_state_dict(guardado["modelo"])
    return modelo, simbolos, {c: i for i, c in enumerate(simbolos)}

## 2. Palabras y conversaciones

In [2]:
# --- Palabras y conversaciones ----------------------------------------------

def preparar(texto, a_id):
    """Divide la novela y elige palabras para entrenar y para la prueba."""
    datos = torch.tensor([a_id[c] for c in texto], dtype=torch.long)
    a, b = int(0.8 * len(datos)), int(0.9 * len(datos))
    veces = {}
    for palabra in re.findall(r"\b[a-záéíóúñ]+\b", texto[:a]):
        if 3 <= len(palabra) <= 8 and all(c in a_id for c in palabra.upper()):
            veces[palabra] = veces.get(palabra, 0) + 1
    palabras = sorted(p for p, n in veces.items() if n >= 3)
    random.Random(SEMILLA).shuffle(palabras)
    corte = int(0.8 * len(palabras))
    return datos[:a], datos[a:b], palabras[:corte], palabras[corte:]


def pregunta(tarea, palabra):
    return f"Usuario: {tarea} «{palabra}»\nAsistente:"


def codificar(tarea, palabra, a_id, contexto):
    """Entrada, objetivos y máscara de una conversación completa."""
    texto_pregunta = pregunta(tarea, palabra)
    respuesta = " " + TAREAS[tarea](palabra) + "\n"
    ids = [a_id[c] for c in texto_pregunta + respuesta]
    mascara = [0] * (len(texto_pregunta) - 1) + [1] * len(respuesta)
    relleno = contexto + 1 - len(ids)
    ids += [a_id["\n"]] * relleno
    mascara += [0] * relleno
    return ids[:-1], ids[1:], mascara


def lote_conversaciones(ejemplos, a_id, contexto, azar, n=24):
    """n conversaciones: cada una, con una tarea y una palabra al azar."""
    filas = []
    for _ in range(n):
        tarea = azar.choice(list(TAREAS))
        palabra = azar.choice(ejemplos[tarea])
        filas.append(codificar(tarea, palabra, a_id, contexto))
    x, y, mascara = zip(*filas)
    return (torch.tensor(x), torch.tensor(y),
            torch.tensor(mascara, dtype=torch.float))


def lote_texto(datos, contexto, generador, n=8):
    maximo = len(datos) - contexto - 1
    inicios = torch.randint(maximo, (n,), generator=generador)
    x = torch.stack([datos[i:i + contexto] for i in inicios])
    y = torch.stack([datos[i + 1:i + contexto + 1] for i in inicios])
    return x, y

In [3]:
torch.set_num_threads(4)
modelo, simbolos, a_id = cargar_modelo()
novela, validacion, palabras, prueba = preparar(TEXTO.read_text(encoding="utf-8"), a_id)
print(len(palabras), "palabras para entrenar y", len(prueba), "de prueba:", palabras[:8])
x, y, mascara = codificar("escribe al revés", "sancho", a_id, 64)
print("Conversación:", repr("".join(simbolos[i] for i in x[:45])))
print("Lo que cuenta en la pérdida:", repr("".join(simbolos[i] for i, m in zip(y, mascara) if m)))

4060 palabras para entrenar y 1016 de prueba: ['diversos', 'acomodó', 'volvióse', 'jarro', 'sepulcro', 'ahogó', 'duelen', 'semejas']
Conversación: 'Usuario: escribe al revés «sancho»\nAsistente:'
Lo que cuenta en la pérdida: ' ohcnas\n'


## 3. Ajuste supervisado

In [4]:
# --- Ajuste supervisado -----------------------------------------------------

def perdida_media(modelo, x, y, mascara=None):
    logits = modelo(x)
    perdidas = F.cross_entropy(logits.view(-1, logits.shape[-1]), y.view(-1),
                               reduction="none")
    if mascara is None:
        return perdidas.mean()
    return (perdidas * mascara.view(-1)).sum() / mascara.sum()


def ajustar(modelo, optimizador, ejemplos, novela, pasos, a_id, azar,
            generador):
    contexto = modelo.config["contexto"]
    for _ in range(pasos):
        x, y, mascara = lote_conversaciones(ejemplos, a_id, contexto, azar)
        perdida = perdida_media(modelo, x, y, mascara)
        if MEZCLAR:  # un poco de novela en cada paso, para no olvidarla
            perdida = perdida + perdida_media(
                modelo, *lote_texto(novela, contexto, generador))
        optimizador.zero_grad(set_to_none=True)
        perdida.backward()
        optimizador.step()

## 4. Responder, recompensar y evaluar

In [5]:
# --- Responder y evaluar ----------------------------------------------------

@torch.no_grad()
def responder(modelo, preguntas, simbolos, a_id, temperatura=0,
              generador=None):
    """Responde a la vez a varias preguntas de la misma longitud."""
    ids = torch.tensor([[a_id[c] for c in p] for p in preguntas])
    for _ in range(11):  # espacio, hasta 8 letras, salto de línea y margen
        logits = modelo(ids)[:, -1]
        if temperatura == 0:
            siguiente = logits.argmax(dim=-1, keepdim=True)
        else:
            probabilidades = F.softmax(logits / temperatura, dim=-1)
            siguiente = torch.multinomial(probabilidades, 1,
                                          generator=generador)
        ids = torch.cat([ids, siguiente], dim=1)
    n = len(preguntas[0])
    textos = ["".join(simbolos[i] for i in fila[n:]) for fila in ids.tolist()]
    return [texto.split("\n")[0].strip() for texto in textos]


def por_longitud(palabras):
    grupos = {}
    for palabra in palabras:
        grupos.setdefault(len(palabra), []).append(palabra)
    return grupos.values()


def recompensa(tarea, palabra, respuesta):
    """1 si la respuesta es correcta y 0 si no."""
    return int(respuesta == TAREAS[tarea](palabra))


def aciertos(modelo, palabras, tarea, simbolos, a_id, temperatura=0,
             intentos=1, generador=None):
    """Fracción de palabras con alguna respuesta premiada en los intentos."""
    bien = 0
    for grupo in por_longitud(palabras):
        preguntas = [pregunta(tarea, palabra) for palabra in grupo]
        premiadas = set()
        for _ in range(intentos):
            respuestas = responder(modelo, preguntas, simbolos, a_id,
                                   temperatura, generador)
            premiadas |= {palabra for palabra, respuesta
                          in zip(grupo, respuestas)
                          if recompensa(tarea, palabra, respuesta)}
        bien += len(premiadas)
    return bien / len(palabras)


@torch.no_grad()
def perdida_novela(modelo, datos, grupo=256):
    """Pérdida media en todo un bloque de la novela (capítulo 7)."""
    T = modelo.config["contexto"]
    n = (len(datos) - 1) // T
    x, y = datos[:n * T].view(n, T), datos[1:n * T + 1].view(n, T)
    total = 0.0
    for i in range(0, n, grupo):
        total += perdida_media(modelo, x[i:i + grupo],
                               y[i:i + grupo]).item() * len(x[i:i + grupo])
    return total / n


def informe(modelo, palabras, validacion, simbolos, a_id):
    resultado = {tarea: aciertos(modelo, palabras, tarea, simbolos, a_id)
                 for tarea in TAREAS}
    resultado["novela"] = perdida_novela(modelo, validacion)
    return resultado


@torch.no_grad()
def continuar(modelo, simbolos, a_id, inicio, n, generador):
    """Continúa un texto sorteando con temperatura 0,8 (capítulo 8)."""
    contexto = modelo.config["contexto"]
    ids = [a_id[c] for c in inicio]
    for _ in range(n):
        logits = modelo(torch.tensor([ids[-contexto:]]))[0, -1]
        probabilidades = F.softmax(logits / 0.8, dim=-1)
        ids.append(int(torch.multinomial(probabilidades, 1,
                                         generator=generador)))
    return "".join(simbolos[i] for i in ids)

## 5. Aprender de los propios aciertos

In [6]:
# --- Recompensas: aprender de los propios aciertos --------------------------

def recoger_aciertos(modelo, palabras, tarea, simbolos, a_id, generador):
    """El modelo responde una vez a cada palabra; guardamos las premiadas."""
    premiadas = []
    for grupo in por_longitud(palabras):
        preguntas = [pregunta(tarea, palabra) for palabra in grupo]
        respuestas = responder(modelo, preguntas, simbolos, a_id, 1.0,
                               generador)
        premiadas += [palabra for palabra, respuesta in zip(grupo, respuestas)
                      if recompensa(tarea, palabra, respuesta)]
    return premiadas

## 6. El experimento completo

Modelo base, ajuste supervisado y tres rondas de recompensas. Tarda unos nueve minutos.

In [7]:
def mostrar(modelo, simbolos, a_id, palabra="dulcinea"):
    for tarea in TAREAS:
        preguntas = [pregunta(tarea, palabra)]
        respuesta = responder(modelo, preguntas, simbolos, a_id)[0]
        print("    %s «%s» -> %s" % (tarea, palabra, respuesta))


def ejecutar(rondas=3):
    torch.set_num_threads(4)
    azar = random.Random(SEMILLA)
    generador = torch.Generator().manual_seed(SEMILLA)
    modelo, simbolos, a_id = cargar_modelo()
    texto = TEXTO.read_text(encoding="utf-8")
    novela, validacion, palabras, prueba = preparar(texto, a_id)
    print("Palabras de entrenamiento y de prueba:", len(palabras), len(prueba))
    resultados = {"base": informe(modelo, prueba, validacion, simbolos, a_id)}
    print("Modelo base:", resultados["base"])
    mostrar(modelo, simbolos, a_id)

    comienzo = time.perf_counter()
    ejemplos = {tarea: palabras for tarea in TAREAS}
    ejemplos["escribe al revés"] = palabras[:DEMOS_REVES]
    optimizador = torch.optim.AdamW(modelo.parameters(), lr=1e-3,
                                    weight_decay=0.1)
    ajustar(modelo, optimizador, ejemplos, novela, 1000, a_id, azar, generador)
    resultados["ajuste"] = informe(modelo, prueba, validacion, simbolos, a_id)
    print("Tras el ajuste supervisado:", resultados["ajuste"])
    mostrar(modelo, simbolos, a_id)
    una, mejor_de_8 = [aciertos(modelo, prueba, "escribe al revés", simbolos,
                                a_id, 1.0, intentos, generador)
                       for intentos in (1, 8)]
    resultados["al revés con T=1"] = [una, mejor_de_8]
    print("Al revés con T=1: una respuesta %.3f · mejor de 8 %.3f"
          % (una, mejor_de_8))

    reves = set(ejemplos["escribe al revés"])
    for ronda in range(1, rondas + 1):
        premiadas = recoger_aciertos(modelo, palabras, "escribe al revés",
                                     simbolos, a_id, generador)
        nuevas = set(premiadas) - reves
        reves |= nuevas
        ejemplos["escribe al revés"] = sorted(reves)
        ajustar(modelo, optimizador, ejemplos, novela, 200, a_id, azar,
                generador)
        etapa = f"ronda {ronda}"
        resultados[etapa] = informe(modelo, prueba, validacion, simbolos, a_id)
        print("Ronda", ronda, "·", len(nuevas), "premiadas nuevas ·",
              len(reves), "ejemplos al revés:", resultados[etapa])
    print("Tiempo desde el ajuste: %.0f s" % (time.perf_counter() - comienzo))
    libre = continuar(modelo, simbolos, a_id, "En un lugar de la Mancha", 150,
                      generador)
    print("Texto libre:", repr(libre))
    return resultados

In [8]:
resultados = ejecutar()

Palabras de entrenamiento y de prueba: 4060 1016
Modelo base: {'repite': 0.0, 'pon en mayúsculas': 0.0, 'escribe al revés': 0.0, 'novela': 1.4775648843457492}
    repite «dulcinea» -> que el cur
    pon en mayúsculas «dulcinea» -> que le dij
    escribe al revés «dulcinea» -> ¿qué de la
Tras el ajuste supervisado: {'repite': 0.906496062992126, 'pon en mayúsculas': 0.9399606299212598, 'escribe al revés': 0.22539370078740156, 'novela': 1.6157652881964792}
    repite «dulcinea» -> dulcinea
    pon en mayúsculas «dulcinea» -> DULCINEA
    escribe al revés «dulcinea» -> aeicilid
Al revés con T=1: una respuesta 0.116 · mejor de 8 0.313
Ronda 1 · 489 premiadas nuevas · 789 ejemplos al revés: {'repite': 0.890748031496063, 'pon en mayúsculas': 0.9389763779527559, 'escribe al revés': 0.37696850393700787, 'novela': 1.6116692681799956}
Ronda 2 · 635 premiadas nuevas · 1424 ejemplos al revés: {'repite': 0.952755905511811, 'pon en mayúsculas': 0.9596456692913385, 'escribe al revés': 0.44094488188976

## Ejercicios

1. Explica con tus palabras las tres fases del entrenamiento de un asistente. ¿Qué datos usa cada una? ¿En cuál de ellas no hace falta saber de antemano la respuesta correcta?

2. Escribe la conversación completa para `pon en mayúsculas` con la palabra `sancho`. ¿Cuántos caracteres tiene, contando los dos saltos de línea? ¿Cuántas predicciones cuentan en la pérdida?

3. Calcula la longitud de la conversación más larga que puede aparecer con palabras de 8 letras. ¿Cabe en el contexto? ¿Qué palabra más larga cabría?

4. ¿Qué aprendería además el modelo si la pérdida contara también los caracteres de la pregunta? ¿Sería un problema en un asistente real?

5. Cambia `MEZCLAR` a `False` y ejecuta el programa. Compara la pérdida en la novela, los aciertos y el texto libre con los del capítulo. ¿Qué ha pasado?

6. Con una sola respuesta sorteada, el modelo acierta al revés el 11,6 % de las palabras de prueba. Si los ocho intentos fueran independientes, ¿qué probabilidad habría de acertar al menos uno? Compárala con el 31,3 % medido. ¿Por qué el valor real es menor?

7. Para comprobar que la mejora de las rondas viene de las recompensas, sustituye `recoger_aciertos` por una función que devuelva siempre una lista vacía y ejecuta el programa. ¿Qué acierto alcanza ahora la tarea al revés? ¿Qué demuestra la comparación?

8. Supón que, en lugar de guardar la palabra, guardáramos la respuesta que escribió el modelo. Propón dos recompensas mal diseñadas para la tarea al revés y explica qué aprendería el modelo con cada una.

9. Añade a `TAREAS` una cuarta tarea, `di la primera letra de`, cuya respuesta sea la primera letra de la palabra, y ejecuta el programa. (La petición debe ser corta: la pregunta y los 11 caracteres de la respuesta tienen que caber en las 64 posiciones del contexto.) ¿La aprende? ¿Afecta a las otras tres?

10. Informe antes y después. Con los resultados de tu ejecución, escribe un informe breve, de unas diez líneas: qué datos se usaron en cada fase, cómo cambió el acierto en cada tarea y la pérdida en la novela, y qué aportaron las recompensas.

## Referencias

Inspirado en [nanochat](https://github.com/karpathy/nanochat), de Andrej Karpathy, y en su vídeo [Deep Dive into LLMs like ChatGPT](https://www.youtube.com/watch?v=7xTGNNLPyMI). Texto de *Don Quijote de la Mancha*, de dominio público. Código y experimentos propios.